In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-GGHq2YjvxskA


In [2]:
# import v1.0
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# LLM 모델
# FewShotPromptTemplate
from langchain_core.prompts.few_shot import FewShotPromptTemplate

## FewShotPromptTemplate 설계

In [4]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },
    {
    "country": "일본에 대해서 어떻게 알고 있나요?",
    "answer": """
        저는 이렇게 알고 있어요
        수도: 도쿄
        언어: 일본어
        음식: 스시와 라멘
        통화: 엔
    """
    },
    {
        "country": "대한민국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요
            수도: 서울
            언어: 한국어
            음식: 김치와 불고기
            통화: 원
        """
    },
    {
    "country": "이탈리아에 대해서 어떻게 알고 있나요?",
    "answer": """
        저는 이렇게 알고 있어요
        수도: 로마
        언어: 이탈리아어
        음식: 피자와 파스타
        통화: 유로
    """
    }
]

In [5]:
example_template = """
    Human: {country}
    AI: {answer}
"""

example_prompt = PromptTemplate.from_template(example_template)

example_prompt

PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='\n    Human: {country}\n    AI: {answer}\n')

## 3단계: FewShotPromptTemplate 생성 후 결합

In [7]:
prompt = FewShotPromptTemplate(
    # 질문
    example_prompt=example_prompt,
    # 예시
    examples=examples,
    # 사용자의 질문 
    suffix="Human: {country}에 대해서 어떻게 알고 있어요?",
    input_variables=["country"]
)

In [8]:
prompt

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'country': '프랑스에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요\n            수도: 파리\n            언어: 프랑스어\n            음식: 와인과 치즈\n            통화: 유로\n        '}, {'country': '일본에 대해서 어떻게 알고 있나요?', 'answer': '\n        저는 이렇게 알고 있어요\n        수도: 도쿄\n        언어: 일본어\n        음식: 스시와 라멘\n        통화: 엔\n    '}, {'country': '대한민국에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요\n            수도: 서울\n            언어: 한국어\n            음식: 김치와 불고기\n            통화: 원\n        '}, {'country': '이탈리아에 대해서 어떻게 알고 있나요?', 'answer': '\n        저는 이렇게 알고 있어요\n        수도: 로마\n        언어: 이탈리아어\n        음식: 피자와 파스타\n        통화: 유로\n    '}], example_prompt=PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='\n    Human: {country}\n    AI: {answer}\n'), suffix='Human: {country}에 대해서 어떻게 알고 있어요?')

In [9]:
chat = ChatOpenAI(temperature=0)

In [10]:
chain = prompt | chat

In [25]:
result = chain.invoke({
    "country": "북극"
})

In [26]:
print(result.content)

AI: 
        저는 이렇게 알고 있어요
        위치: 북극 지역
        특징: 극지점에 위치하고 극한의 추위와 얼음이 많이 있는 지역
        동식물: 북극곰, 펭귄 등이 서식
        환경: 급격한 기후 변화로 인해 얼음이 녹는 문제가 심각함


## FewShotChatMessagePromptTemplate

In [27]:
# v1.0
# ChatModel
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate

In [28]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },
    {
    "country": "일본에 대해서 어떻게 알고 있나요?",
    "answer": """
        저는 이렇게 알고 있어요
        수도: 도쿄
        언어: 일본어
        음식: 스시와 라멘
        통화: 엔
    """
    },
    {
        "country": "대한민국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요
            수도: 서울
            언어: 한국어
            음식: 김치와 불고기
            통화: 원
        """
    },
    {
    "country": "이탈리아에 대해서 어떻게 알고 있나요?",
    "answer": """
        저는 이렇게 알고 있어요
        수도: 로마
        언어: 이탈리아어
        음식: 피자와 파스타
        통화: 유로
    """
    }
]

## 2단계 예시용 프롬프트 템플릿 정의

In [64]:
example = ChatPromptTemplate.from_messages([
    ("human", "{country}에 대해서 어떻게 알고 있어요?"),
    ("ai", "{answer}"),
])

In [65]:
example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example,
    examples=examples
)

In [66]:
final = prompt | chain

In [67]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 지리학 전문가입니다. 짧은 답변을 제공하세요."),
    example_prompt,
    ("human", "{country}에 대해서 어떻게 알고 있어요?")
])

In [68]:
print(final_prompt)

input_variables=['country'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 지리학 전문가입니다. 짧은 답변을 제공하세요.'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'country': '프랑스에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요\n            수도: 파리\n            언어: 프랑스어\n            음식: 와인과 치즈\n            통화: 유로\n        '}, {'country': '일본에 대해서 어떻게 알고 있나요?', 'answer': '\n        저는 이렇게 알고 있어요\n        수도: 도쿄\n        언어: 일본어\n        음식: 스시와 라멘\n        통화: 엔\n    '}, {'country': '대한민국에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요\n            수도: 서울\n            언어: 한국어\n            음식: 김치와 불고기\n            통화: 원\n        '}, {'country': '이탈리아에 대해서 어떻게 알고 있나요?', 'answer': '\n        저는 이렇게 알고 있어요\n        수도: 로마\n        언어: 이탈리아어\n        음식: 피자와 파스타\n        통화: 유로\n    '}], input_variables=[], input_types={}, partial_variables={}, ex

In [69]:
final_chain = final_prompt | chat

In [70]:
result = final_chain.invoke({
    "country": "일본"
})

In [71]:
print(result.content)

일본은 아시아 대륙 동쪽에 위치한 섬나라로, 수도는 도쿄입니다. 일본어가 공용어로 사용되며, 일본은 전통적인 문화와 현대화된 도시들이 공존하는 나라입니다. 일본은 일본식 생활과 예술, 기술력, 자연 경치, 그리고 다양한 음식으로 유명합니다. 일본의 통화는 엔이며, 일본의 교통은 철도와 고속도로가 잘 발달되어 있습니다. 일본은 세계적으로 유명한 관광지와 문화적인 명소들이 많이 있습니다.


## LengthBasedExampleSelector

In [72]:
# v1.0
from langchain_core.example_selectors.length_based import LengthBasedExampleSelector

In [73]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },
    {
    "country": "일본에 대해서 어떻게 알고 있나요?",
    "answer": """
        저는 이렇게 알고 있어요
        수도: 도쿄
        언어: 일본어
        음식: 스시와 라멘
        통화: 엔
    """
    },
    {
        "country": "대한민국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요
            수도: 서울
            언어: 한국어
            음식: 김치와 불고기
            통화: 원
        """
    },
    {
    "country": "이탈리아에 대해서 어떻게 알고 있나요?",
    "answer": """
        저는 이렇게 알고 있어요
        수도: 로마
        언어: 이탈리아어
        음식: 피자와 파스타
        통화: 유로
    """
    }
]

In [74]:
example_prompt = PromptTemplate.from_template("Human: {country}\nAI: {answer}")

## Selector 연결

In [79]:
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    # 예제 양 허용 (토큰 개수)  
    max_length=100
)

In [80]:
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: {country}에 대해서 어떻게 알고 있나요?",
    input_variables=["country"]
)

In [81]:
prompt.format(**{
    "country": "브라질"
})

'Human: 프랑스에 대해서 어떻게 알고 있나요?\nAI: \n            저는 이렇게 알고 있어요\n            수도: 파리\n            언어: 프랑스어\n            음식: 와인과 치즈\n            통화: 유로\n        \n\nHuman: 브라질에 대해서 어떻게 알고 있나요?'

In [82]:
final_chain = prompt | chat

In [83]:
result = final_chain.invoke({
    "country": "독일"
})

In [84]:
print(result.content)

AI: 
            저는 이렇게 알고 있어요
            수도: 베를린
            언어: 독일어
            음식: 소세지와 맥주
            통화: 유로


## Chat 모델(LengthBasedExampleSelector)

In [85]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },
    {
    "country": "일본에 대해서 어떻게 알고 있나요?",
    "answer": """
        저는 이렇게 알고 있어요
        수도: 도쿄
        언어: 일본어
        음식: 스시와 라멘
        통화: 엔
    """
    },
    {
        "country": "대한민국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요
            수도: 서울
            언어: 한국어
            음식: 김치와 불고기
            통화: 원
        """
    },
    {
    "country": "이탈리아에 대해서 어떻게 알고 있나요?",
    "answer": """
        저는 이렇게 알고 있어요
        수도: 로마
        언어: 이탈리아어
        음식: 피자와 파스타
        통화: 유로
    """
    }
]